In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("swiggy.csv")

# Select only required columns
df_clean = df[['name', 'city', 'rating', 'rating_count', 'cost', 'cuisine']].copy()

# Drop duplicate restaurant names (keep first)
df_clean = df_clean.drop_duplicates(subset=['name'], keep='first')


# Clean 'rating' column
# Set '--' to 3.0 in rating, then convert to numeric
df_clean['rating'] = df_clean['rating'].replace('--', 3.0)
df_clean['rating'] = pd.to_numeric(df_clean['rating'], errors='coerce')

# Map 'rating_count'
rating_map = {
    'Too Few Ratings': 10.0,
    '50+ ratings': 50.0,
    '100+ ratings': 100.0,
    '20+ ratings': 20.0,
    '500+ ratings': 500.0,
    '1K+ ratings': 1000.0,
    '5K+ ratings': 5000.0,
    '10K+ ratings': 10000.0,
}
df_clean['rating_count'] = df_clean['rating_count'].map(rating_map)

# Drop rows with NaN in 'rating_count'
df_clean = df_clean.dropna(subset=['rating_count'])

# Clean 'cost' column
df_clean['cost'] = df_clean['cost'].astype(str).str.replace('₹', '', regex=False)
df_clean['cost'] = df_clean['cost'].str.replace(',', '', regex=False)
df_clean['cost'] = pd.to_numeric(df_clean['cost'], errors='coerce')

# Drop duplicates
df_clean = df_clean.drop_duplicates()

# # Fill missing numerical values with median
df_clean = df_clean.dropna(subset=['cost'])

# Drop rows with missing text fields
df_clean = df_clean.dropna(subset=['name', 'city', 'cuisine'])

# Save cleaned file
df_clean.to_csv("cleaned_data.csv", index=False)


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
import pickle
import joblib

# Load cleaned data
df = pd.read_csv("cleaned_data.csv")

### --------- City Encoding ---------
# Split 'city' into 'sub_city' and 'main_city'
df[['sub_city', 'main_city']] = df['city'].str.split(',', n=1, expand=True)

# Handle missing values
df['sub_city'] = df['sub_city'].fillna('Unknown').str.strip()
df['main_city'] = df['main_city'].fillna(df['sub_city']).str.strip()

# One-hot encode 'main_city' and 'sub_city'
city_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_cities = city_encoder.fit_transform(df[['main_city', 'sub_city']])
encoded_city_df = pd.DataFrame(encoded_cities, columns=city_encoder.get_feature_names_out(['main_city', 'sub_city']), index=df.index)

# Drop original city fields
df = df.drop(columns=['city', 'main_city', 'sub_city'])

# Concatenate city encoded data
df = pd.concat([df, encoded_city_df], axis=1)

# Save city-encoded DataFrame
df.to_csv("city_encoded.csv", index=False)
joblib.dump(city_encoder, "city_encoder.joblib")

### --------- Cuisine Encoding ---------
# Check if 'cuisine' column exists
if 'cuisine' in df.columns:
    df['cuisine'] = df['cuisine'].str.split(',').apply(lambda x: [i.strip() for i in x])
    cuisine_encoder = MultiLabelBinarizer()
    cuisine_encoded = cuisine_encoder.fit_transform(df['cuisine'])
    cuisine_encoded_df = pd.DataFrame(cuisine_encoded, columns=cuisine_encoder.classes_, index=df.index)
    
    # Drop original cuisine column
    df = df.drop(columns=['cuisine'])
    
    # Add encoded cuisine columns
    df = pd.concat([df, cuisine_encoded_df], axis=1)
    
    print("Cuisine column encoded and dropped.")
    joblib.dump(cuisine_encoder, "cuisine_encoder.joblib")
else:
    print("The 'cuisine' column does not exist in the dataframe.")

### --------- Final Save ---------
# Save fully encoded DataFrame
df.to_csv("encoded_data.csv", index=False)

# Save entire encoded DataFrame as joblib
joblib.dump(df, "encoded_data.joblib")

# Optional: Load to verify
df_loaded = joblib.load("encoded_data.joblib")
print("Encoded data loaded successfully from joblib.")

### --------- Optional Index Alignment ---------
cleaned_data = pd.read_csv("cleaned_data.csv", index_col=False)
encoded_data = pd.read_csv("encoded_data.csv", index_col=False)

# Aligning indices
encoded_data_aligned = encoded_data.loc[encoded_data.index.isin(cleaned_data.index)]
cleaned_data_aligned = cleaned_data.loc[cleaned_data.index.isin(encoded_data.index)]

# Check if aligned
print("Index alignment check:", encoded_data_aligned.index.equals(cleaned_data_aligned.index))


Cuisine column encoded and dropped.
Encoded data loaded successfully from joblib.
Index alignment check: True


In [3]:
import pandas as pd
from sklearn.cluster import KMeans
import numpy as np

# Load the encoded data (all features must be numerical)
encoded_data = pd.read_csv("encoded_data.csv", index_col=0)

# Function for K-Means Clustering
def kmeans_clustering(data, n_clusters=5):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    data['Cluster'] = kmeans.fit_predict(data)
    return data, kmeans

# Perform clustering
encoded_data_with_clusters, kmeans_model = kmeans_clustering(encoded_data.copy(), n_clusters=5)

# Load the cleaned data
cleaned_data = pd.read_csv("cleaned_data.csv", index_col=0)

# Add cluster info to cleaned data
cleaned_data['Cluster'] = encoded_data_with_clusters['Cluster']

# ✅ Confirm index alignment
print("Index aligned:", cleaned_data.index.equals(encoded_data_with_clusters.index))

# 🎯 Example: Get recommendations for a specific restaurant by position
restaurant_position = 0  # Change this to any index you want to test

# Get the cluster of the selected restaurant
restaurant_cluster = cleaned_data.iloc[restaurant_position]['Cluster']

# Get all restaurant indices in the same cluster (excluding the selected one)
same_cluster_indices = cleaned_data[cleaned_data['Cluster'] == restaurant_cluster].index.tolist()
selected_index = cleaned_data.index[restaurant_position]
if selected_index in same_cluster_indices:
    same_cluster_indices.remove(selected_index)

# Top 5 similar restaurants
top_5_recommendations = cleaned_data.loc[same_cluster_indices[:5]]

# Show the recommended restaurants
print("Top 5 similar restaurants:")
print(top_5_recommendations)


c:\Users\Sriraman\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Sriraman\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Sriraman\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Sriraman\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, c

Index aligned: True
Top 5 similar restaurants:
                     city  rating  rating_count   cost  \
name                                                     
Janta Sweet House  Abohar     4.4          50.0  200.0   
theka coffee desi  Abohar     3.8         100.0  100.0   
Singh Hut          Abohar     3.7          20.0  250.0   
GRILL MASTERS      Abohar     3.0          10.0  250.0   
Sam Uncle          Abohar     3.6          20.0  200.0   

                                      cuisine  Cluster  
name                                                    
Janta Sweet House               Sweets,Bakery        0  
theka coffee desi                   Beverages        0  
Singh Hut                    Fast Food,Indian        0  
GRILL MASTERS      Italian-American,Fast Food        0  
Sam Uncle                         Continental        0  
